# Assumption 1: Linearity in Parameters ($Y = X\beta + \epsilon$)
**Prepared for:** Assumptions of Least Square Regressions Lecture Suite  
**Author:** Nick Lim

Ordinary Least Squares (OLS) requires that the regression model is **linear in parameters (`Assumption 1`)**. Mathematically, the expected value of the outcome variable must be a linear combination of the unknown parameters $\beta_0, \beta_1, \dots, \beta_k$:
$$Y_i = \beta_0 + \beta_1 X_{1i} + \beta_2 X_{2i} + \dots + \epsilon_i$$

### What does 'Linearity in Parameters' actually mean?
It means that each parameter $\beta_j$ enters the equation as a simple multiplier. Crucially, the model **does not need to be linear in the predictors ($X$)**! For example, polynomial regressions ($Y = \beta_0 + \beta_1 X + \beta_2 X^2$) and logarithmic models ($Y = \beta_0 + \beta_1 \ln X$) are 100% linear in parameters and fully satisfy Assumption 1.

### What happens when Assumption 1 is violated?
If the true underlying data-generating process is non-linear (e.g., cubic or exponential) and we fit a naive linear model ($Y = \beta_0 + \beta_1 X$):
1. Our coefficient estimates become severely biased underfitters.
2. The diagnostic **`Residuals vs Fitted` plot exhibits a distinct U-shaped or S-shaped systematic pattern** instead of random noise around zero.
3. Applying the correct transformation (such as $X^3$ or $\ln X$) immediately restores exact linearity in parameters, eliminating the residual pattern.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

if not hasattr(matplotlib.rcParams, '_get'):
    matplotlib.rcParams._get = lambda key: matplotlib.rcParams[key]

np.random.seed(42)
sns.set_theme(style="whitegrid")
plt.rcParams["font.size"] = 12

## 1. Simulating Cubic Curvature ($Y = -5 + 0.8 X^3 + \epsilon$)

Let's simulate $n = 100$ data points spanning $X \in [-3, 3]$ where the true relationship is strictly cubic.

In [ ]:
n = 100
x = np.linspace(-3.0, 3.0, n)
true_error = np.random.normal(0, 3.5, n)
y = -5.0 + 0.8 * (x ** 3) + true_error

df = pd.DataFrame({"x": x, "y": y, "x3": x ** 3})

# Model 1: Naive Linear OLS (Violating Assumption 1 specification)
mod_naive = sm.OLS(df["y"], sm.add_constant(df["x"])).fit()

# Model 2: Transformed Polynomial OLS (Restoring Linearity in Parameters)
mod_trans = sm.OLS(df["y"], sm.add_constant(df["x3"])).fit()

print("=== Model Comparison Summary ===")
comp_df = pd.DataFrame({
    "Specification": ["Naive Linear (y ~ x)", "Transformed Polynomial (y ~ x^3)"],
    "Estimated Intercept": [mod_naive.params[0], mod_trans.params[0]],
    "Estimated Slope": [mod_naive.params[1], mod_trans.params[1]],
    "R-Squared": [mod_naive.rsquared, mod_trans.rsquared],
    "AIC": [mod_naive.aic, mod_trans.aic]
})
print(comp_df.to_string(index=False))

## 2. Visual Diagnostic: Fits vs. Residual Curves

Let's plot side-by-side diagnostic charts to observe how the `Residuals vs Fitted` chart catches specification error right away.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Top-Left: Naive Linear Fit on Scatter Plot
axes[0, 0].scatter(df["x"], df["y"], color="#64748b", alpha=0.6, label="Observed Data")
axes[0, 0].plot(df["x"], mod_naive.fittedvalues, color="#dc2626", linewidth=3.0, label="Naive Linear OLS Fit ($R^2=0.74$)")
axes[0, 0].set_title("Naive Linear Fit ($Y = \beta_0 + \beta_1 X$)")
axes[0, 0].set_xlabel("Predictor ($X$)")
axes[0, 0].set_ylabel("Outcome ($Y$)")
axes[0, 0].legend()

# Top-Right: Naive Residuals vs Fitted (Shows S-Shaped Curvature)
axes[0, 1].scatter(mod_naive.fittedvalues, mod_naive.resid, color="#dc2626", alpha=0.7)
axes[0, 1].axhline(0, color="#1e293b", linestyle="--", linewidth=1.5)
# Overlay smoothed trend line
sns.regplot(x=mod_naive.fittedvalues, y=mod_naive.resid, scatter=False, lowess=True, color="#1e293b", ax=axes[0, 1], line_kws={"linewidth": 2.5})
axes[0, 1].set_title("Diagnostic: Naive Residuals vs Fitted Values")
axes[0, 1].set_xlabel("Fitted Values ($\hat{Y}$)")
axes[0, 1].set_ylabel("Residuals ($e_i$)")

# Bottom-Left: Transformed Polynomial Fit
axes[1, 0].scatter(df["x"], df["y"], color="#64748b", alpha=0.6, label="Observed Data")
axes[1, 0].plot(df["x"], mod_trans.fittedvalues, color="#2563eb", linewidth=3.0, label="Transformed OLS ($Y = \beta_0 + \beta_1 X^3$)")
axes[1, 0].set_title("Transformed Polynomial Fit ($R^2=0.96$)")
axes[1, 0].set_xlabel("Predictor ($X$)")
axes[1, 0].set_ylabel("Outcome ($Y$)")
axes[1, 0].legend()

# Bottom-Right: Transformed Residuals vs Fitted (Clean & Random Noise)
axes[1, 1].scatter(mod_trans.fittedvalues, mod_trans.resid, color="#2563eb", alpha=0.7)
axes[1, 1].axhline(0, color="#1e293b", linestyle="--", linewidth=1.5)
sns.regplot(x=mod_trans.fittedvalues, y=mod_trans.resid, scatter=False, lowess=True, color="#1e293b", ax=axes[1, 1], line_kws={"linewidth": 2.5})
axes[1, 1].set_title("Diagnostic: Transformed Residuals vs Fitted")
axes[1, 1].set_xlabel("Fitted Values ($\hat{Y}$)")
axes[1, 1].set_ylabel("Residuals ($e_i$)")

plt.tight_layout()
plt.show()

## Key Takeaways
1. **Residual Curvature is the #1 Red Flag:** If your `Residuals vs Fitted` scatter plot shows systematic bowing or S-shaped LOESS trends, Assumption 1 is violated.
2. **Transformation Fixes Specification:** Creating new terms ($X^2, X^3, \ln X$) or interaction terms ($X_1 \times X_2$) allows standard linear OLS to fit complex non-linear curves while keeping exact linearity in parameters.